# Credit Card Default: Evaluation and Explainability

A bank wants to predict which of its credit card clients will default next month, to refuse or reduce their credit, and to be able to explain each decision, to the client as well as to the regulator: creditworthiness assessment is a high-risk system under the AI Act. The data: 30,000 clients of a Taiwanese bank in 2005 ([Yeh & Lien, 2009](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients), CC BY 4.0), 22% of whom defaulted.

1. A first model and its areas under the curves
2. The decision threshold against a cost matrix
3. Probability calibration
4. Global explanation with SHAP
5. A badly predicted client against a well predicted one
6. LIME against SHAP on the same client
7. Permutation importance and partial dependence
8. Data slices: where the model costs the most
9. Fairness: errors by group

The notebook runs without a GPU, in a few minutes.

## Data

Each row is a client: `limit` (the credit granted, in New Taiwan dollars), `sex` (1 = male, 2 = female), `education` (1 = graduate school, 2 = university, 3 = high school, 4 = other; 0, 5 and 6 are undocumented), `marriage`, `age`, then for each month from April to September 2005: `delay_<month>` (the payment delay, in months; -1: paid duly; -2 and 0, undocumented, aren't delays), `bill_<month>` (the bill statement's amount) and `paid_<month>` (the amount paid). `default` is 1 if the client defaulted the next month. The [dataset's README](https://github.com/shuuchuu/datasets/tree/main/credit-default) details the columns.

In [ ]:
# Colab lacks lime and fairlearn; its shap 0.52 rejects a background set with xgboost 3
%pip install -q lime fairlearn "shap>=0.53"
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh credit-default

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

clients = pd.read_csv("credit-default/credit-default.csv", index_col="id")
# As floats: scikit-learn's partial dependence refuses integer columns.
X = clients.drop(columns="default").astype(float)
y = clients["default"]
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.25, stratify=y, random_state=0
)
print(
  f"{len(X_train)} training clients, {len(X_test)} test clients, "
  f"{y.mean():.1%} defaults"
)
clients.head()

## 1. A First Model

Train `model`, an `XGBClassifier`, on `X_train` and `y_train`, and compute `proba`, the default probability of each client of `X_test`. Compare the model's accuracy at the 0.5 threshold with that of a model that never predicts a default. Then compute the ROC AUC and the PR AUC: what value, that of chance, should each be compared with? Finally, plot both curves side by side.

**Useful documentation**

- [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)`: 300 shallow trees, learned slowly; `random_state` makes the training reproducible
- `model.predict_proba(X_test)[:, 1]`: the column of class 1, the default
- `accuracy_score(y_test, proba > 0.5)`; a model that never predicts a default has, as its accuracy, the share of healthy clients, `1 - y_test.mean()`
- `roc_auc_score(y_test, proba)` and `average_precision_score(y_test, proba)`: chance is 0.5 for the first, the prevalence `y_test.mean()` for the second
- `fig, axes = plt.subplots(1, 2, figsize=(10, 4))`, then `RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])`, and likewise `PrecisionRecallDisplay` on `axes[1]`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.metrics import (
  PrecisionRecallDisplay,
  RocCurveDisplay,
  accuracy_score,
  average_precision_score,
  roc_auc_score,
)
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)
model.fit(..., ...)
proba = model.predict_proba(X_test)[:, ...]

print(f"Model accuracy: {accuracy_score(y_test, ...):.3f}")
print(f"Accuracy of 'never a default': {...:.3f}")
print(f"ROC AUC: {roc_auc_score(y_test, proba):.3f} (chance: 0.5)")
print(f"PR AUC: {...:.3f} (chance: {y_test.mean():.3f})")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(..., ..., ax=...)
plt.show()
```

### Solution

In [ ]:
from sklearn.metrics import (
  PrecisionRecallDisplay,
  RocCurveDisplay,
  accuracy_score,
  average_precision_score,
  roc_auc_score,
)
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

print(f"Model accuracy: {accuracy_score(y_test, proba > 0.5):.3f}")
print(f"Accuracy of 'never a default': {1 - y_test.mean():.3f}")
print(f"ROC AUC: {roc_auc_score(y_test, proba):.3f} (chance: 0.5)")
print(
  f"PR AUC: {average_precision_score(y_test, proba):.3f} (chance: {y_test.mean():.3f})"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1])
plt.show()

## 2. The Decision Threshold against a Cost Matrix

Refusing credit to a client who would have repaid it (a false positive) loses a client; granting credit to a client who defaults (a false negative) costs five times as much. Write `cost(y_true, y_pred)`, the total cost of the errors, a false positive costing `C_FP` and a false negative `C_FN`. Compare the cost on the test set:

- at the 0.5 threshold;
- at the threshold `threshold` the slides' rule gives for calibrated probabilities, $C_\text{FP} / (C_\text{FP} + C_\text{FN})$;
- at the threshold `TunedThresholdClassifierCV` picks by cross-validation on the training set.

**Useful documentation**

- [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [`make_scorer`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html)

In [ ]:
C_FP, C_FN = 1, 5  # the cost of a false positive, that of a false negative


def cost(y_true, y_pred) -> int: ...


# Your code here

### Additional hints

The calls and the parameters to use:

- `confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()` returns `tn, fp, fn, tp`; `labels=[0, 1]` keeps a 2 × 2 matrix even on a group of clients without any default
- An example to check by hand: `cost([0, 1, 1], [1, 0, 1])` is 6 (a false positive, a false negative)
- `proba > threshold`: the refusals at a given threshold
- `make_scorer(cost, greater_is_better=False)`: a score to minimize
- `TunedThresholdClassifierCV(estimator, scoring=...)`, with `estimator` a fresh model, `XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)`, then `.fit(X_train, y_train)` (five trainings, about ten seconds); `tuned.best_threshold_` and `tuned.predict(X_test)`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import TunedThresholdClassifierCV

C_FP, C_FN = 1, 5


def cost(y_true, y_pred) -> int:
  _, fp, fn, _ = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
  return ...


print(f"Cost at the 0.5 threshold: {cost(y_test, proba > 0.5)}")
threshold = ...
print(f"Cost at the {threshold:.3f} threshold: {cost(y_test, proba > threshold)}")

tuned = TunedThresholdClassifierCV(
  XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0),
  scoring=make_scorer(..., greater_is_better=...),
)
tuned.fit(X_train, y_train)
print(
  f"Cost at the cross-validated threshold ({tuned.best_threshold_:.3f}): "
  f"{cost(y_test, tuned.predict(X_test))}"
)
```

### Solution

In [ ]:
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import TunedThresholdClassifierCV

C_FP, C_FN = 1, 5


def cost(y_true, y_pred) -> int:
  _, fp, fn, _ = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
  return int(C_FP * fp + C_FN * fn)


print(f"Cost at the 0.5 threshold: {cost(y_test, proba > 0.5)}")
threshold = C_FP / (C_FP + C_FN)
print(f"Cost at the {threshold:.3f} threshold: {cost(y_test, proba > threshold)}")

tuned = TunedThresholdClassifierCV(
  XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0),
  scoring=make_scorer(cost, greater_is_better=False),
)
tuned.fit(X_train, y_train)
print(
  f"Cost at the cross-validated threshold ({tuned.best_threshold_:.3f}): "
  f"{cost(y_test, tuned.predict(X_test))}"
)

## 3. Calibration

The threshold rule assumes calibrated probabilities: among the clients the model gives 0.2, 20% must default. Plot the reliability diagram of `proba` and compute its Brier score. Then train `balanced`, the same model, with its classes rebalanced by `scale_pos_weight` (the number of healthy clients divided by the number of defaults of `y_train`), and compare both models: their curve, their Brier score, their mean probability against the actual default rate, and their cost at the `threshold` threshold, then at 0.5. What threshold would the rebalanced model need?

**Useful documentation**

- [`CalibrationDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibrationDisplay.html#sklearn.calibration.CalibrationDisplay.from_predictions)
- [`brier_score_loss`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.brier_score_loss.html)
- [`scale_pos_weight`](https://xgboost.readthedocs.io/en/stable/parameter.html#parameters-for-tree-booster)

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `(y_train == 0).sum() / (y_train == 1).sum()`: about 3.5
- `XGBClassifier(..., scale_pos_weight=ratio)`, with the other parameters of `model`
- `CalibrationDisplay.from_predictions(y_test, proba, n_bins=10, name="XGBoost", ax=ax)`: the diagonal is perfect calibration; a second call on the same `ax` adds the other model
- `brier_score_loss(y_test, proba)`: the mean squared error of the probabilities, smaller is better

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss

ratio = ...
balanced = XGBClassifier(
  n_estimators=300,
  learning_rate=0.05,
  max_depth=4,
  random_state=0,
  scale_pos_weight=...,
)
balanced.fit(X_train, y_train)
balanced_proba = balanced.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [("XGBoost", proba), ("rebalanced", balanced_proba)]:
  CalibrationDisplay.from_predictions(y_test, p, n_bins=10, name=name, ax=ax)
  print(
    f"{name}: Brier {...:.3f}, mean probability {p.mean():.3f}, "
    f"cost {cost(y_test, p > threshold)} at {threshold:.3f}, "
    f"{cost(y_test, p > 0.5)} at 0.5"
  )
print(f"Actual default rate: {y_test.mean():.3f}")
plt.show()
```

### Solution

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss

ratio = (y_train == 0).sum() / (y_train == 1).sum()
balanced = XGBClassifier(
  n_estimators=300,
  learning_rate=0.05,
  max_depth=4,
  random_state=0,
  scale_pos_weight=ratio,
)
balanced.fit(X_train, y_train)
balanced_proba = balanced.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [("XGBoost", proba), ("rebalanced", balanced_proba)]:
  CalibrationDisplay.from_predictions(y_test, p, n_bins=10, name=name, ax=ax)
  print(
    f"{name}: Brier {brier_score_loss(y_test, p):.3f}, mean probability {p.mean():.3f}, "
    f"cost {cost(y_test, p > threshold)} at {threshold:.3f}, "
    f"{cost(y_test, p > 0.5)} at 0.5"
  )
print(f"Actual default rate: {y_test.mean():.3f}")
plt.show()

## 4. Global Explanation with SHAP

Create `explainer`, a `TreeExplainer` of `model` whose background set is a sample of 100 training clients, and compute `explanation`, the SHAP values of `sample`, the first 1,000 test clients (TreeSHAP with a background set takes about ten seconds on them). Check additivity: the base value plus the sum of a client's SHAP values gives the model's output, in log-odds. Show the beeswarm and the bar plot: which features matter most, and in which direction does `delay_sep` act?

**Useful documentation**

- [`shap.TreeExplainer`](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html)
- [`DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)

In [ ]:
sample = X_test.iloc[:1000]

# Your code here

### Additional hints

The calls and the parameters to use:

- `shap.TreeExplainer(model, X_train.sample(100, random_state=0))`: the background set is the second argument
- `explainer(sample)` returns a `shap.Explanation`: `.values` (a row per client, a column per feature) and `.base_values` (the base value, the model's mean output on the background set, the same for all)
- `model.predict(sample, output_margin=True)`: the model's output in log-odds; the largest gap, `np.abs(...).max()`, should be around 1e-6
- `shap.plots.beeswarm(explanation)` and `shap.plots.bar(explanation)`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import shap

sample = X_test.iloc[:1000]
explainer = shap.TreeExplainer(model, ...)
explanation = explainer(...)
margin = model.predict(sample, output_margin=True)
gap = np.abs(explanation.base_values + ... - margin).max()
print(
  f"Base value: {explanation.base_values[0]:.3f} log-odds; "
  f"largest additivity gap: {gap:.1e}"
)
shap.plots.beeswarm(...)
shap.plots.bar(...)
```

### Solution

In [ ]:
import shap

sample = X_test.iloc[:1000]
explainer = shap.TreeExplainer(model, X_train.sample(100, random_state=0))
explanation = explainer(sample)
margin = model.predict(sample, output_margin=True)
gap = np.abs(explanation.base_values + explanation.values.sum(axis=1) - margin).max()
print(
  f"Base value: {explanation.base_values[0]:.3f} log-odds; "
  f"largest additivity gap: {gap:.1e}"
)
shap.plots.beeswarm(explanation)
shap.plots.bar(explanation)

## 5. A Badly Predicted Client against a Well Predicted One

Among the test clients who defaulted, find `worst`, the one the model deemed the least risky, and `best`, the one it deemed the riskiest: their positions in `X_test`. Show their features side by side, then the SHAP waterfall of each. Why was the model wrong about `worst`? Does the error come from the model (a default it could have seen in these features) or from the data (nothing in its features tells it apart from a good payer)?

**Useful documentation**

- [`numpy.flatnonzero`](https://numpy.org/doc/stable/reference/generated/numpy.flatnonzero.html)

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `np.flatnonzero(y_test == 1)`: the positions of the clients who defaulted; `proba[defaulters].argmin()` gives, among them, that of the least risky according to the model
- `X_test.iloc[[worst, best]].T`: their features, a column per client
- `explainer(X_test.iloc[[worst, best]])`: the SHAP values of both clients; `shap.plots.waterfall(cases[0])`, then `cases[1]`
- The waterfall starts from the base value, at the bottom, and adds each feature's contribution up to the model's output for this client, at the top

A skeleton to copy into the exercise cell, replacing each `...`:

```python
defaulters = np.flatnonzero(y_test == 1)
worst = defaulters[proba[defaulters].argmin()]
best = ...
print(f"Default probability: {proba[worst]:.3f} for worst, {proba[best]:.3f} for best")
display(X_test.iloc[[worst, best]].T)
cases = explainer(...)
shap.plots.waterfall(cases[0])
shap.plots.waterfall(...)
```

### Solution

In [ ]:
defaulters = np.flatnonzero(y_test == 1)
worst = defaulters[proba[defaulters].argmin()]
best = defaulters[proba[defaulters].argmax()]
print(f"Default probability: {proba[worst]:.3f} for worst, {proba[best]:.3f} for best")
display(X_test.iloc[[worst, best]].T)
cases = explainer(X_test.iloc[[worst, best]])
shap.plots.waterfall(cases[0])
shap.plots.waterfall(cases[1])

## 6. LIME against SHAP on the Same Client

Explain the prediction of `worst` with LIME: a `LimeTabularExplainer` built on `X_train`, then `explain_instance` with the 5 most important features. Compare with this client's 5 largest SHAP values, in absolute value. Run LIME again with another `random_state`: does the explanation change? Why don't LIME and SHAP give the same numbers?

**Useful documentation**

- [`LimeTabularExplainer`](https://lime-ml.readthedocs.io/en/latest/lime.html#lime.lime_tabular.LimeTabularExplainer)

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `LimeTabularExplainer(X_train.to_numpy(), feature_names=list(X.columns), class_names=["healthy", "default"], mode="classification", random_state=0)`: LIME works on NumPy arrays, and cuts each feature into quartiles
- `lime_explainer.explain_instance(X_test.to_numpy()[worst], model.predict_proba, num_features=5)`; `.as_list()` returns the conditions and their weights, on the default probability
- `pd.Series(cases[0].values, index=X.columns)`: the SHAP values of `worst` (computed in exercise 5); `shap_values[shap_values.abs().nlargest(5).index]`: the 5 largest
- LIME fits a local linear model, on features cut into intervals, to the probability; SHAP splits the output in log-odds exactly between the features

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from lime.lime_tabular import LimeTabularExplainer

for seed in (0, 1):
  lime_explainer = LimeTabularExplainer(
    X_train.to_numpy(),
    feature_names=list(X.columns),
    class_names=["healthy", "default"],
    mode="classification",
    random_state=seed,
  )
  lime_explanation = lime_explainer.explain_instance(
    ..., model.predict_proba, num_features=...
  )
  print(f"LIME, random_state={seed}:")
  for condition, weight in lime_explanation.as_list():
    print(f"  {condition:30} {weight:+.3f}")

shap_values = pd.Series(cases[0].values, index=X.columns)
print("SHAP:")
print(shap_values[...].round(3))
```

### Solution

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

for seed in (0, 1):
  lime_explainer = LimeTabularExplainer(
    X_train.to_numpy(),
    feature_names=list(X.columns),
    class_names=["healthy", "default"],
    mode="classification",
    random_state=seed,
  )
  lime_explanation = lime_explainer.explain_instance(
    X_test.to_numpy()[worst], model.predict_proba, num_features=5
  )
  print(f"LIME, random_state={seed}:")
  for condition, weight in lime_explanation.as_list():
    print(f"  {condition:30} {weight:+.3f}")

shap_values = pd.Series(cases[0].values, index=X.columns)
print("SHAP:")
print(shap_values[shap_values.abs().nlargest(5).index].round(3))

## 7. Permutation Importance and Partial Dependence

Compute the permutation importance of each feature on the test set, as the drop of the ROC AUC, and compare its ranking with that of SHAP's bar plot. Then plot the partial dependence and the ICE curves of `delay_sep` and `limit`: is the effect of `limit` the same for every client?

**Useful documentation**

- [`PartialDependenceDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.PartialDependenceDisplay.html#sklearn.inspection.PartialDependenceDisplay.from_estimator)

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=0)` (about fifteen seconds); `pd.Series(result.importances_mean, index=X.columns).nlargest(8)`: the 8 most important
- `PartialDependenceDisplay.from_estimator(model, X_test, features=["delay_sep", "limit"], kind="both", subsample=300, random_state=0)`: `kind="both"` plots the ICE curves, one per client, and their mean, the partial dependence; `subsample=300` keeps only 300 clients, so that the curves stay readable

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

result = permutation_importance(
  model, X_test, y_test, scoring=..., n_repeats=10, random_state=0
)
print(pd.Series(result.importances_mean, index=X.columns).nlargest(8).round(4))

PartialDependenceDisplay.from_estimator(
  model,
  X_test,
  features=[..., ...],
  kind=...,
  subsample=300,
  random_state=0,
)
plt.show()
```

### Solution

In [ ]:
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

result = permutation_importance(
  model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=0
)
print(pd.Series(result.importances_mean, index=X.columns).nlargest(8).round(4))

PartialDependenceDisplay.from_estimator(
  model,
  X_test,
  features=["delay_sep", "limit"],
  kind="both",
  subsample=300,
  random_state=0,
)
plt.show()

## 8. Data Slices: Where the Model Costs the Most

From now on, the decisions are those of the cross-validated threshold (`decisions`: 1 for a refusal). Write `slice_report(slices)`, which returns a table with one row per slice of the test clients: its proportion, its default rate, the recall (the share of defaults refused), the refusal rate, the mean cost per client and the potential impact. Credit has no human-level performance to compare with: the gap is measured here against a perfect model, at zero cost, so it is the cost per client; the potential impact is, as in the slides, the gap times the slice's proportion. Apply `slice_report` to the September delay (`delay`), to age (`age`) and to education (`education`). Which slice should be worked on first? What does it have in common with the client `worst`?

**Useful documentation**

- [`DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [`recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)

In [ ]:
decisions = pd.Series(tuned.predict(X_test), index=X_test.index)
delay = pd.cut(
  X_test.delay_sep, [-3, 0, 1, 9], labels=["up to date", "1 month", "2 months or more"]
)
age = pd.cut(
  X_test.age, [20, 30, 40, 50, 80], labels=["21-30", "31-40", "41-50", "51+"]
)
education = X_test.education.map(
  {1: "graduate school", 2: "university", 3: "high school"}
).fillna("other or unknown")


def slice_report(slices: pd.Series) -> pd.DataFrame: ...


# Your code here

### Additional hints

The calls and the parameters to use:

- `pd.DataFrame({"y": y_test, "refused": decisions, "slice": slices}).groupby("slice", observed=True)`: a group per slice; `observed=True` skips empty categories
- For each `(name, group)` of the `groupby`: `len(group) / len(data)`, `group.y.mean()`, `recall_score(group.y, group.refused)`, `group.refused.mean()`, `cost(group.y, group.refused) / len(group)`
- A dictionary of rows, `{name: {"proportion": ..., ...}}`, becomes a table with `pd.DataFrame.from_dict(rows, orient="index")`
- To check: the potential impact of the "up to date" slice is about 0.46

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.metrics import recall_score


def slice_report(slices: pd.Series) -> pd.DataFrame:
  data = pd.DataFrame({"y": y_test, "refused": decisions, "slice": slices})
  rows = {}
  for name, group in data.groupby("slice", observed=True):
    proportion = ...
    per_client = cost(group.y, group.refused) / len(group)
    rows[name] = {
      "proportion": proportion,
      "default rate": group.y.mean(),
      "recall": ...,
      "refusal rate": ...,
      "cost per client": per_client,
      "impact": ...,
    }
  return pd.DataFrame.from_dict(rows, orient="index").round(3)


for slices in (delay, age, education):
  display(slice_report(slices))
```

### Solution

In [ ]:
from sklearn.metrics import recall_score


def slice_report(slices: pd.Series) -> pd.DataFrame:
  data = pd.DataFrame({"y": y_test, "refused": decisions, "slice": slices})
  rows = {}
  for name, group in data.groupby("slice", observed=True):
    proportion = len(group) / len(data)
    per_client = cost(group.y, group.refused) / len(group)
    rows[name] = {
      "proportion": proportion,
      "default rate": group.y.mean(),
      "recall": recall_score(group.y, group.refused),
      "refusal rate": group.refused.mean(),
      "cost per client": per_client,
      "impact": per_client * proportion,
    }
  return pd.DataFrame.from_dict(rows, orient="index").round(3)


for slices in (delay, age, education):
  display(slice_report(slices))

## 9. Fairness: Errors by Group

With a Fairlearn `MetricFrame`, compare between men and women the recall, the refusal rate and the false positive rate of the decisions (the share of healthy clients refused), then do the same by age band (`age`). What gaps do you see, and which one weighs on clients who would have repaid? `sex` is one of the model's features: would removing it be enough to remove these gaps?

**Useful documentation**

- [`false_positive_rate`](https://fairlearn.org/main/api_reference/generated/fairlearn.metrics.false_positive_rate.html)

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `X_test.sex.map({1: "male", 2: "female"})`: readable group names
- `MetricFrame(metrics={"recall": recall_score, ...}, y_true=y_test, y_pred=decisions, sensitive_features=sex)`: `metrics` maps a name to each `(y_true, y_pred)` function; `selection_rate` and `false_positive_rate` come from `fairlearn.metrics`
- `frame.by_group`: a row per group; `frame.difference()`: the largest gap between groups, for each metric

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from fairlearn.metrics import MetricFrame, false_positive_rate, selection_rate

sex = X_test.sex.map({1: "male", 2: "female"})
metrics = {
  "recall": recall_score,
  "refusal rate": ...,
  "false positives": ...,
}
for groups in (sex, age):
  frame = MetricFrame(
    metrics=metrics, y_true=y_test, y_pred=..., sensitive_features=...
  )
  display(frame.by_group.round(3))
  print(frame.difference().round(3))
```

### Solution

In [ ]:
from fairlearn.metrics import MetricFrame, false_positive_rate, selection_rate

sex = X_test.sex.map({1: "male", 2: "female"})
metrics = {
  "recall": recall_score,
  "refusal rate": selection_rate,
  "false positives": false_positive_rate,
}
for groups in (sex, age):
  frame = MetricFrame(
    metrics=metrics, y_true=y_test, y_pred=decisions, sensitive_features=groups
  )
  display(frame.by_group.round(3))
  print(frame.difference().round(3))

## Going Further

- A model interpretable by design, often as good as XGBoost on tabular data: [InterpretML's EBM](https://interpret.ml/docs/ebm.html) (`ExplainableBoostingClassifier`, `pip install interpret`)
- Counterfactuals, the smallest changes that would have had a refused client's credit granted, touching only the features they can act on: [DiCE](https://github.com/interpretml/DiCE) (`pip install dice-ml`)